# Lecture 7 Demo: Build an ISP, Then Break It

**Instructor demo.** About twenty seconds, no external files.

Assemble the pipeline stage by stage, compare two demosaicing methods we wrote ourselves,
then reorder the stages and watch a real number fall.

Because the scene is synthetic we know the ground truth, so "better" is measured rather
than asserted.

In [ ]:
import sys
from pathlib import Path

# Put the shared slide helpers on the path, wherever this notebook is opened from.
for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (parent / "psu_imaging.py").exists():
        sys.path.insert(0, str(parent))
        break

import cv2
import matplotlib.pyplot as plt
import numpy as np
import psu_imaging as psu

plt.rcParams.update(psu.figure_style())
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(513)
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

In [ ]:
from pathlib import Path
here = Path.cwd().resolve()
sys.path.insert(0, str(next(p for p in [here, *here.parents] if (p / "make_figures.py").exists())))
from make_figures import bilinear_demosaic, mhc_demosaic, opencv_demosaic

scene = psu.build_scene()
truth = psu.linear_to_srgb(scene)
blurred = psu.apply_lens(scene, 0.8) * psu.CHANNEL_GAIN
raw = psu.expose_and_digitize(psu.mosaic_rggb(blurred), np.random.default_rng(3))
print(f"raw: {raw.shape[1]} x {raw.shape[0]}, {raw.dtype}, {raw.min()} to {raw.max()} DN")

## 1. The stages, in order

Five stages is enough to make a recognizable photograph. A shipping ISP has tens, but
the shape is this.

In [ ]:
GAINS = np.array([1 / 0.55, 1.0, 1 / 0.62])          # white balance, from lecture 6

def black_level(raw, black=psu.BLACK_LEVEL, white=psu.WHITE_LEVEL):
    return np.clip((raw.astype(float) - black) / (white - black), 0, 1)

def white_balance_mosaic(m, gains=GAINS):
    """Applied in the mosaic, before interpolation, so demosaicing sees balanced data."""
    x = m.copy()
    x[0::2, 0::2] *= gains[0]      # R
    x[0::2, 1::2] *= gains[1]      # G on red rows
    x[1::2, 0::2] *= gains[1]      # G on blue rows
    x[1::2, 1::2] *= gains[2]      # B
    return np.clip(x, 0, 1)

def run_isp(raw, demosaic=mhc_demosaic, subtract_black=True, gamma_before_matrix=False,
            ccm=None):
    m = black_level(raw) if subtract_black else np.clip(raw.astype(float) / psu.WHITE_LEVEL, 0, 1)
    m = white_balance_mosaic(m)
    rgb = demosaic(m)
    if ccm is not None:
        if gamma_before_matrix:
            rgb = np.clip(psu.linear_to_srgb(rgb) @ np.asarray(ccm).T, 0, 1)
            return rgb
        rgb = np.clip(rgb @ np.asarray(ccm).T, 0, 1)
    return psu.linear_to_srgb(rgb)


stages = {}
m = black_level(raw);                       stages["1. black level"] = m.copy()
m = white_balance_mosaic(m);                stages["2. white balance"] = m.copy()
rgb = mhc_demosaic(m);                      stages["3. demosaic"] = rgb.copy()
stages["4. gamma"] = psu.linear_to_srgb(rgb)

fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, (name, img) in zip(axes, stages.items()):
    ax.imshow(np.clip(img, 0, 1), cmap=None if img.ndim == 3 else "gray")
    ax.set_title(name, fontsize=9.5); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Two demosaicing methods we wrote

Bilinear interpolates each channel from its own neighbors. Malvar-He-Cutler adds a
Laplacian of the channel that *was* measured at that pixel, on the argument that
luminance detail is shared between channels.

In [ ]:
wb = white_balance_mosaic(black_level(raw))
methods = {"bilinear": bilinear_demosaic(wb),
           "Malvar-He-Cutler": mhc_demosaic(wb),
           "OpenCV edge aware": opencv_demosaic(wb)}

print(f"{'method':22s} {'PSNR (dB)':>10}")
for name, img in methods.items():
    print(f"{name:22s} {psu.psnr(psu.linear_to_srgb(img), truth):10.2f}")

ry, rx, rn = 320, 480, 110
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, (name, img) in zip(axes, methods.items()):
    ax.imshow(psu.linear_to_srgb(img)[ry:ry + rn, rx:rx + rn]); ax.set_title(name, fontsize=9.5)
axes[3].imshow(truth[ry:ry + rn, rx:rx + rn]); axes[3].set_title("ground truth", fontsize=9.5)
for ax in axes: ax.axis("off")
plt.suptitle("Into the frequency sweep, where the detail is finer than the color filter array",
             fontsize=10.5, y=1.02)
plt.tight_layout(); plt.show()

## 3. Zippering

Along a near-vertical edge, alternating rows interpolate differently and the edge develops
a comb. This is the artifact the gradient correction exists to fix.

In [ ]:
ry, rx, rn = 330, 200, 60
fig, axes = plt.subplots(2, 3, figsize=(9.5, 6))
for j, (img, name) in enumerate([(psu.linear_to_srgb(methods["bilinear"]), "bilinear"),
                                 (psu.linear_to_srgb(methods["Malvar-He-Cutler"]), "MHC"),
                                 (truth, "truth")]):
    axes[0, j].imshow(img[ry:ry + rn, rx:rx + rn]); axes[0, j].set_title(name, fontsize=10)
    axes[1, j].imshow(np.abs(img - truth).mean(axis=2)[ry:ry + rn, rx:rx + rn],
                      cmap="magma", vmin=0, vmax=0.15)
    axes[1, j].set_title("error", fontsize=9)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Now reorder it

Same stages, different order. One of these costs more than every demosaicing improvement
put together.

In [ ]:
# Fit the colour matrix to THIS camera from the chart patches, so that every variant
# below runs the same pipeline and differs only in stage order. An arbitrary matrix
# would make this comparison meaningless.
from make_figures import CFA_CROSSTALK

blurred_x = (psu.apply_lens(scene, 0.8) @ CFA_CROSSTALK.T) * psu.CHANNEL_GAIN
raw_x = psu.expose_and_digitize(psu.mosaic_rggb(blurred_x), np.random.default_rng(3))

lin = mhc_demosaic(white_balance_mosaic(black_level(raw_x)))
half, ph, pw = scene.shape[0] // 2, scene.shape[0] // 8, scene.shape[1] // 6
src, dst = [], []
for i in range(24):
    r, c = divmod(i, 6)
    sl = (slice(r * ph + ph // 4, r * ph + 3 * ph // 4),
          slice(c * pw + pw // 4, c * pw + 3 * pw // 4))
    src.append(lin[sl].reshape(-1, 3).mean(axis=0))
    dst.append(scene[sl].reshape(-1, 3).mean(axis=0))
CCM, *_ = np.linalg.lstsq(np.array(src), np.array(dst), rcond=None)
CCM = CCM.T
print("fitted colour matrix (camera RGB -> scene RGB):")
print(np.array2string(CCM, precision=3, suppress_small=True))

variants = {
    "correct order":               run_isp(raw_x, ccm=CCM),
    "black level never subtracted": run_isp(raw_x, subtract_black=False, ccm=CCM),
    "gamma before the matrix":     run_isp(raw_x, gamma_before_matrix=True, ccm=CCM),
    "sharpen before denoise":      run_isp(raw_x, ccm=CCM),      # replaced below
    "bilinear instead of MHC":     run_isp(raw_x, demosaic=bilinear_demosaic, ccm=CCM),
}
del variants["sharpen before denoise"]

print()
print(f"{'pipeline':32s} {'PSNR (dB)':>10}  {'cost vs correct':>16}")
base = psu.psnr(variants["correct order"], truth)
for name, img in variants.items():
    p_ = psu.psnr(img, truth)
    print(f"{name:32s} {p_:10.2f}  {p_ - base:+16.2f}")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, (name, img) in zip(axes, variants.items()):
    ax.imshow(np.clip(img[60:210, 60:210], 0, 1)); ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. A convention, measured

The usual advice is to white balance **before** demosaicing, because demosaicing assumes
the channels are correlated. Let us check it rather than repeat it.

In [ ]:
before = run_isp(raw_x, ccm=CCM)
after_rgb = np.clip(mhc_demosaic(black_level(raw_x)) * GAINS, 0, 1)
after = psu.linear_to_srgb(np.clip(after_rgb @ CCM.T, 0, 1))

print(f"white balance before demosaic  {psu.psnr(before, truth):.2f} dB")
print(f"white balance after  demosaic  {psu.psnr(after, truth):.2f} dB")
print()
print("The convention holds here. My first version of this experiment said the opposite,")
print("because the simulated camera had no cross-talk between colour channels. With")
print("independent channels there is no inter-channel correlation for demosaicing to")
print("exploit, so the ordering could not matter. A simulation that omits the mechanism")
print("cannot test the claim that depends on it.")

## Back to the slides

Land on this: forgetting one subtraction cost more than the entire difference between a
naive demosaicer and a good one. Order encodes assumptions, and the assumptions are where
the bugs live.

Project 2 is this notebook, on your own raw files, with a learned denoiser added.